# Import Packages

In [1]:
# Import Packages
%load_ext autoreload
%autoreload 2

import random

from dice_model import face_lookup, mcp_dice
from resolution import (
    apply_deterministic_mod,
    apply_reroll,
)
from rules import (
    DiceModQuantity,
    PassiveRule,
    PropertyOverride,
    Window,
    pierce_rule,
    reroll_x_rule,
)

# Baron Helmut Zemo vs. Miles Morales

Baron Helmut Zemo (BHZ) attacks Miles Morales (MM) with 5 attack dice; MM has 3 defense dice.

## Special Rules for Zemo:

- Pierce - If the attack roll contains at least one *Wild* results, change one of the defender's *Critical*, *Wild*.or *Shield* results to a *Blank*.
- BHZ may reroll 1 die in his attack roll.
- BHZ may reroll 3 dice in his attack roll.

In [2]:
zemo_pierce = pierce_rule(
    name="Pierce",
    window=Window.ATTACKER_MOD_DEFENDER,
    target_faces=["Critical", "Wild", "Shield"],
    quantity=DiceModQuantity(
        max_quantity=1,
        faces_to_count=["Wild"],
        matching_rule="all",
        source_roll="own"
    )
)

zemo_reroll_1 = reroll_x_rule(
    name="Zemo Reroll 1",
    window=Window.ATTACKER_MOD_SELF,
    max_quantity=1
)

zemo_reroll_3 = reroll_x_rule(
    name="Zemo Reroll 3",
    window=Window.ATTACKER_MOD_SELF,
    max_quantity=3
)

## Special Rules for Miles:

- MM can modify *Skull* results in his defense rolls.
- MM may reroll 2 dice in his defense roll.
- MM may reroll 1 die in his defense roll.

In [3]:
miles_skulls_modifiable = PassiveRule(
    name="Modify Skulls",
    override=PropertyOverride(
        face="Skull",
        property="is_modifiable",
        effect=True
    )
)

miles_reroll_1 = reroll_x_rule(
    name="Miles Reroll 1",
    window=Window.DEFENDER_MOD_SELF,
    max_quantity=1,
)

miles_reroll_2 = reroll_x_rule(
    name="Miles Reroll 2",
    window=Window.DEFENDER_MOD_SELF,
    max_quantity=2,
)

# Create Dice Pools (Steps 4 & 5) 

In [4]:
# Step 4 - Create Attackers Dice Pool
attacker_pool = 5

# Step 5 - Create Defenders Dice Pool
defender_pool = 3

# Roll Initial Dice Pool (Steps 6 & 7)

In [5]:
# Step 6 - Roll the attacker's dice pool
attacker_original_roll = random.choices(mcp_dice, k=attacker_pool)

# Step 7 - Rolle the defenders' dice pool
defender_original_roll = random.choices(mcp_dice, k=defender_pool)

# Resolve Criticals (Step 8)

In [6]:
attacker_additional_pool = sum(
    face_lookup[face].additional_dice
    for face in attacker_original_roll
)

attacker_additional_roll = random.choices(
    mcp_dice, 
    k=attacker_additional_pool
)

attacker_full_roll = attacker_original_roll + attacker_additional_roll

defender_additional_pool = sum(
    face_lookup[face].additional_dice
    for face in defender_original_roll
)

defender_additional_roll = random.choices(
    mcp_dice, 
    k=defender_additional_pool
)

defender_full_roll = defender_original_roll + defender_additional_roll

# Attacker Modifies Self (Step 9.a.i)

In [7]:
attacker_current_roll = attacker_full_roll.copy()
attack_current_results = []
attacker_track_reroll = []

for result in attacker_current_roll:
    attack_current_results.append(
        face_lookup[result].attack_value
    )

for rule in [zemo_reroll_3,zemo_reroll_1]:
    attacker_current_roll = apply_reroll(
        roll=attacker_current_roll,
        results=attack_current_results,
        rule=rule,
        passive_rules=[]
        )

    attack_current_results = []

    for result in attacker_current_roll:
        attack_current_results.append(
        face_lookup[result].attack_value
    )
    
    attacker_track_reroll.append(attacker_current_roll)

# Defender Modify Self (Step 9.a.ii)

In [8]:
defender_current_roll = defender_full_roll.copy()
defend_current_results = []
defender_track_reroll = []

for result in defender_current_roll:
    defend_current_results.append(
        face_lookup[result].defense_value
    )

for rule in [miles_reroll_2,miles_reroll_1]:
    defender_current_roll = apply_reroll(
        roll=defender_current_roll,
        results=defend_current_results,
        rule=rule,
        passive_rules=[miles_skulls_modifiable]
        )

    defend_current_results = []

    for result in defender_current_roll:
        defend_current_results.append(
        face_lookup[result].defense_value
    )
    
    defender_track_reroll.append(defender_current_roll)

# Calculate Results After Self Mods

In [9]:
attacker_results = []
defender_results = []

for result in attacker_current_roll:
    attacker_results.append(
    face_lookup[result].attack_value
    )

for result in defender_current_roll:
    defender_results.append(
        face_lookup[result].defense_value
    )


# Attacker Mods Defender (Step 9.b.i)

In [10]:
# apply Zemo pierce
defender_modded_roll = apply_deterministic_mod(
    target_roll=defender_current_roll,
    target_results=defender_results,
    rule=zemo_pierce,
    own_roll=attacker_current_roll
    ,opponent_roll=defender_current_roll
    ,passive_rules=[miles_skulls_modifiable] 
)


# Calculate Success or Failure (Step 10)

In [11]:
attacker_results = []
defender_results = []

for result in attacker_current_roll:
    attacker_results.append(
    face_lookup[result].attack_value
    )

for result in defender_modded_roll:
    defender_results.append(
        face_lookup[result].defense_value
    )

attack_successes = sum(attacker_results)
defend_succeses = sum(defender_results)
attack_damage = max(0,attack_successes-defend_succeses)

if attack_damage > 0:
    attack_result = f"Attack did {attack_damage} damage"
else:
    attack_result = "Attack did no damage"

In [12]:
print("Attacker Dice")
print(f"    Original roll:    {attacker_original_roll}")
print(f"    Exploded roll:    {attacker_additional_roll}")
print(f"    Full roll:        {attacker_full_roll}")
print(f"    Self mod roll:    {attacker_current_roll}")
print(f"    Reroll history:   {attacker_track_reroll}")

print("\n")

print("Defender Dice")
print(f"    Original roll:    {defender_original_roll}")
print(f"    Exploded roll:    {defender_additional_roll}")
print(f"    Full roll:        {defender_full_roll}")
print(f"    Self mod roll:    {defender_current_roll}")
print(f"    Opponent mod roll:  {defender_modded_roll}")
print(f"    Reroll history:   {defender_track_reroll}")

print("\n")
print(f"Attacker successes: {sum(attacker_results)}")
print(f"Defender successes: {sum(defender_results)}")
print(attack_result)


Attacker Dice
    Original roll:    ['Hit', 'Wild', 'Blank', 'Critical', 'Hit']
    Exploded roll:    ['Critical']
    Full roll:        ['Hit', 'Wild', 'Blank', 'Critical', 'Hit', 'Critical']
    Self mod roll:    ['Hit', 'Wild', 'Critical', 'Critical', 'Hit', 'Critical']
    Reroll history:   [['Hit', 'Wild', 'Critical', 'Critical', 'Hit', 'Critical'], ['Hit', 'Wild', 'Critical', 'Critical', 'Hit', 'Critical']]


Defender Dice
    Original roll:    ['Blank', 'Blank', 'Blank']
    Exploded roll:    []
    Full roll:        ['Blank', 'Blank', 'Blank']
    Self mod roll:    ['Wild', 'Critical', 'Blank']
    Opponent mod roll:  ['Blank', 'Critical', 'Blank']
    Reroll history:   [['Wild', 'Critical', 'Blank'], ['Wild', 'Critical', 'Blank']]


Attacker successes: 6
Defender successes: 1
Attack did 5 damage
